In [ ]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate


# Map evaluation folder name suffixes (set by exp_name in the run scripts) to
# paper-style display names. Each suffix appears between the experiment prefix
# (e.g. "H384_1e6steps_") and the trailing "_<ode_t_steps>steps". Order
# matters: longer keys are checked first so "projection_all_gradient_guidance"
# wins over "projection_all".
DISPLAY_NAMES = [
    ("projection_all_gradient_guidance", "Projection-All + Gradient Guidance"),
    ("projection_late_gradient_guidance", "Projection-Late + Gradient Guidance"),
    ("projection_relaxed_gradient_guidance", "Projection-Relaxed + Gradient Guidance"),
    ("projection_all", "Projection-All"),
    ("projection_late", "Projection-Late"),
    ("projection_relaxed", "Projection-Relaxed"),
    ("hardflow_new", "HardFlow (l4casadi-free)"),
    ("hardflow", "HardFlow"),
    ("oc_flow", "OC-Flow"),
    ("gradient_guidance", "Gradient Guidance"),
    ("original", "Original"),
]


def get_display_name(experiment_name: str) -> str:
    for key, name in DISPLAY_NAMES:
        if key in experiment_name:
            return name
    return experiment_name


def analyze_experiment_results(env_name, filter_name=[None]):
    base_path = Path("../logs") / env_name / "eval"

    results = []

    for exp_dir in base_path.glob("*"):
        if not exp_dir.is_dir():
            continue

        csv_file = exp_dir / "trajectories.csv"
        if not csv_file.exists():
            continue

        not_match = False
        for n in filter_name:
            if n is not None and n not in exp_dir.name:
                not_match = True
                break
        if not_match:
            continue

        df_traj = pd.read_csv(csv_file)

        if df_traj.empty:
            continue

        total_trajs = len(df_traj)
        safety_trajs = df_traj["safety"].sum()
        safety_rate = safety_trajs / total_trajs if total_trajs > 0 else 0.0

        total_violations_mean = df_traj["total_violations"].mean()
        total_violations_std = (
            df_traj["total_violations"].std() if len(df_traj) > 1 else 0.0
        )

        score_mean = df_traj["score"].mean()
        score_std = df_traj["score"].std() if len(df_traj) > 1 else 0.0

        steps_mean = df_traj["steps"].mean()
        steps_std = df_traj["steps"].std() if len(df_traj) > 1 else 0.0

        if "average_computation_time" in df_traj.columns:
            computation_time_values = df_traj["average_computation_time"].dropna()
            if len(computation_time_values) > 0:
                computation_time_mean = computation_time_values.mean()
                computation_time_std = (
                    computation_time_values.std()
                    if len(computation_time_values) > 1
                    else 0.0
                )
                computation_time_available = True
            else:
                computation_time_mean = None
                computation_time_std = None
                computation_time_available = False
        else:
            computation_time_mean = None
            computation_time_std = None
            computation_time_available = False

        clean_name = exp_dir.name
        clean_name = clean_name.replace("H384_1e6steps_", "")
        display_name = get_display_name(clean_name)

        results.append(
            {
                "experiment": display_name,
                "safety_rate": safety_rate,
                "total_violations_mean": total_violations_mean,
                "total_violations_std": total_violations_std,
                "score_mean": score_mean,
                "score_std": score_std,
                "steps_mean": steps_mean,
                "steps_std": steps_std,
                "computation_time_mean": computation_time_mean,
                "computation_time_std": computation_time_std,
                "computation_time_available": computation_time_available,
                "total_trajs": total_trajs,
                "safety_trajs": safety_trajs,
            }
        )

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values("safety_rate", ascending=False)

    return df


def display_results_table(df, header):
    if df.empty:
        print(f"\n{header} - No experiments found\n")
        return

    table_data = []
    for _, row in df.iterrows():
        if (
            row["computation_time_available"]
            and row["computation_time_mean"] is not None
        ):
            computation_time_str = (
                f"{row['computation_time_mean']:.3f}±{row['computation_time_std']:.3f}"
            )
        else:
            computation_time_str = "N/A"

        total_violations_str = (
            f"{row['total_violations_mean']:.2f}±{row['total_violations_std']:.2f}"
        )
        score_str = f"{row['score_mean']:.2f}±{row['score_std']:.2f}"
        steps_str = f"{row['steps_mean']:.2f}±{row['steps_std']:.2f}"

        table_data.append(
            [
                row["experiment"],
                f"{row['safety_rate']:.2f}",
                total_violations_str,
                score_str,
                steps_str,
                computation_time_str,
                f"{row['total_trajs']}",
            ]
        )

    headers = [
        "Experiment",
        "Safety Rate",
        "Violations",
        "Score",
        "Steps",
        "Computation Time (s)",
        "Total Trials",
    ]

    print(f"\n{header} - {len(df)} experiments:")
    print(
        tabulate(
            table_data,
            headers=headers,
            tablefmt="grid",
            stralign="left",
            disable_numparse=True,
        )
    )


for env_name in ["maze2d-large-v1"]:
    results_df = analyze_experiment_results(env_name, filter_name=[""])
    if results_df.empty:
        continue

    print(f"\n{'=' * 120}")
    print(f"Results for {env_name}".center(120))
    print(f"{'=' * 120}")
    print(f"\nSummary: {len(results_df)} experiments")
    display_results_table(results_df, "Maze Navigation")
    print(f"\n{'=' * 120}")